# Supplementary Table 2

Full-route shuffled-control acceptance and control exposure (v28d legend line 477). The notebook rebuilds the published 18-sheet workbook from `source_data/STab2/`, compares every cell with the published file in `expected/`, recomputes the acquisition quartiles of the `Quantile_summary` sheet, and recomputes the two counts that the Methods cite to this table (v28d line 743).

Run from the repository root: `python paper.py build STab2`.

In [ ]:
from pathlib import Path
import sys, json
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rebuilt workbook; gitignored
OUT.mkdir(parents=True, exist_ok=True)
CHECKS = UNIT / "checks"
CHECKS.mkdir(exist_ok=True)
import pandas as pd
import stab2_table as t

## Table

All 18 sheets in published order. `Cohort_overview` is computed from the per-acquisition, stage and baseline tables (medians across acquisitions at peptide q ≤ 0.01, lengths 5–50; statistical unit acquisition, n = 15, 58, 306, 331). The other sheets are the analysis tables as written by the original producers. The rebuilt workbook is compared cell by cell with `expected/Supplementary_Table_2.xlsx` (numbers compared exactly).

In [ ]:
written = t.write_workbook(SD, OUT / "Supplementary_Table_2.xlsx")
for name, n in written:
    print(f"{name:34s} {n:>7d} rows")
res = t.compare_workbooks(OUT / "Supplementary_Table_2.xlsx", UNIT / "expected" / "Supplementary_Table_2.xlsx")
(CHECKS / "table_compare.json").write_text(json.dumps(res, indent=1) + "\n")
print("sheet order equal:", res["sheet_order_equal"])
for name, s in res["sheets"].items():
    print(f"{name:34s} differing cells: {s.get('differing_cells')}")
print("IDENTICAL" if res["identical"] else "DIFFERENT")

## Quantile summary

The `Quantile_summary` sheet holds, per cohort, q threshold, length floor and control class, the median, first and third quartiles (`statistics.quantiles`, method `exclusive`) and maximum of the per-acquisition fractions. These are acquisition quartiles, not confidence intervals. Recomputed here from `per_acquisition.tsv` and compared with the table shipped in the workbook.

In [ ]:
qs = t.quantile_summary(SD)
src = pd.read_csv(SD / "quantile_summary.tsv", sep="\t")
keys = ["cohort", "control_class", "metric"]
num = ["q_threshold", "min_residues", "n_attempted", "n_defined", "median", "q1", "q3", "max"]
assert (qs[keys].values == src[keys].values).all()
dev = (qs[num].astype(float) - src[num].astype(float)).abs().max()
print(dev.to_string())
print("max absolute deviation:", float(dev.max()))

## Methods line 743

"Shuffle-only peptides accounted for 2 of 376,412 accepted peptide–acquisition observations with acquisition-specific databases and 12 of 411,354 with the other-eleven unions (Supplementary Table 2)." (v28d line 743)

Twelve healthy-stool technical-replicate acquisitions per arm, peptide q ≤ 0.01, lengths 5–50; counts summed over acquisitions (statistical unit: peptide–acquisition observation). The source is the healthy-stool pooling experiment (`hstool_pooling_per_acquisition.tsv`), which is not a sheet of the published workbook; the last lines test whether either denominator occurs anywhere in it.

In [ ]:
s743 = t.stat_line743(SD)
for arm, v in s743.items():
    print(arm, v)
hits = t.find_in_workbook(UNIT / "expected" / "Supplementary_Table_2.xlsx", [376412, 411354])
print("cells in the published workbook equal to the denominators:", hits)

## Numbers

Every recomputed number in `NUMBERS.tsv` order.

In [ ]:
nums = pd.read_csv(UNIT / "NUMBERS.tsv", sep="\t", dtype=str)
values = {
    "2": s743["own"]["shuffle_only"], "376,412": s743["own"]["accepted"],
    "12": s743["pool_other11"]["shuffle_only"], "411,354": s743["pool_other11"]["accepted"],
}
for r in nums.itertuples():
    print(r.v28d_line, r.printed_value, values[r.printed_value], r.status)